In [ ]:
# load dataset
import pandas as pd

df = pd.read_csv("dataset.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()  # drop NA if any

In [ ]:
# preprocessing
# Select duplicate rows
duplicated_df = df[df.duplicated()]
len(duplicated_df)

In [ ]:
duplicated_df

In [ ]:
# before dedup
len(df)

In [ ]:
df = df.drop_duplicates()

In [ ]:
# after dedup
len(df)

In [ ]:
df["label"].unique()

In [ ]:
# data manipulation
# Labels: 1 = bullying, 0 = non-bullying
def perform_data_manipulation(df):
    df = df.copy()
    df["label"] = df["label"].replace(-1, 1)
    return df

In [ ]:
df = perform_data_manipulation(df)

In [ ]:
df["label"].unique()

In [ ]:
df["label"].value_counts()

In [ ]:
# preprocessing

import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import contractions

# Build stopword set — REMOVE negation words so they are preserved
stop = set(stopwords.words("english"))
negation_words = {
    "not",
    "no",
    "never",
    "neither",
    "nor",
    "nothing",
    "nowhere",
    "nobody",
    "none",
    "cannot",
    "without",
    "against",
    "hardly",
    "scarcely",
    "barely",
    "doesnt",
    "isnt",
    "wasnt",
    "shouldnt",
    "wouldnt",
    "couldnt",
    "wont",
    "cant",
    "dont",
    "didnt",
    "hadnt",
    "hasnt",
    "havent",
    "neednt",
    "mightnt",
    "mustnt",
}
stop = stop - negation_words


def expand_contractions(text):
    """Expand contractions so negations are preserved: don't → do not."""
    return contractions.fix(text)


def negate_sequence(text):
    """
    Attach NOT_ prefix to meaningful words (non-stopwords) that follow
    a negation term, until a clause boundary is reached.
    E.g. 'do not hate you' → 'do not NOT_hate NOT_you'
    """
    negation_tokens = {
        "not",
        "no",
        "never",
        "nobody",
        "nothing",
        "nowhere",
        "neither",
        "nor",
        "cannot",
        "without",
        "hardly",
        "scarcely",
        "barely",
    }
    clause_breakers = {"but", "however", "although", "though", "yet"}

    tokens = text.split()
    result = []
    negating = False

    for token in tokens:
        # Strip trailing punctuation for matching, keep original for output
        clean_token = token.rstrip(".,!?;:")

        if clean_token in negation_tokens:
            negating = True
            result.append(token)
        elif clean_token in clause_breakers or token.endswith((".", "!", "?", ";")):
            negating = False
            result.append(token)
        elif negating and clean_token not in stop:
            result.append("NOT_" + clean_token)
        else:
            result.append(token)

    return " ".join(result)


def preprocess_text(text):
    wl = WordNetLemmatizer()

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Expand contractions FIRST so "don't" → "do not" before any removal
    text = expand_contractions(text)

    # Remove emojis
    emoji_clean = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002702-\U000027b0"
        "\U000024c2-\U0001f251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_clean.sub(r"", text)

    # Clean up spacing and URLs
    text = re.sub(r"\.(?=\S)", ". ", text)
    text = re.sub(r"http\S+", "", text)

    # Lowercase, keep only letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())

    # Apply negation tagging BEFORE stopword removal
    text = negate_sequence(text)

    # Remove stopwords and lemmatize
    tokens = []
    for word in text.split():
        if word.startswith("NOT_"):
            root = word[4:]  # strip NOT_ for lemmatization
            if root.isalpha():
                tokens.append("NOT_" + wl.lemmatize(root))
        elif word not in stop and word.isalpha():
            tokens.append(wl.lemmatize(word))

    return " ".join(tokens)

In [ ]:
df["headline"][0]

In [ ]:
df["content"] = df["headline"].apply(preprocess_text)

In [ ]:
df["content"][0]

In [ ]:
# quick test to check negation
sanity_tests = [
    "Nobody likes you",
    "I do not hate you",
    "You are never good enough",
    "I never want to hurt you",
    "Don't be so mean",
    "I don't think you're stupid",
]
print("\n── Negation Sanity Check ──")
for t in sanity_tests:
    print(f"  Original : {t}")
    print(f"  Processed: {preprocess_text(t)}\n")

In [ ]:
words_len = df["content"].str.split().map(lambda x: len(x))
df_temp = df.copy()
df_temp["words length"] = words_len
df_temp

In [ ]:
max(df_temp["words length"])

In [ ]:
from collections import Counter

words = " ".join(df["content"]).split()
counter = Counter(words)

# remove stopwords
filtered = {word: count for word, count in counter.items() if word not in stop}

# most common
most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)

# least common
least_common = sorted(filtered.items(), key=lambda x: x[1])

print("Most common:", most_common[:20])
print("Least common:", least_common[:20])

In [ ]:
# Text encoding
from sklearn.model_selection import train_test_split

In [ ]:
x_data = df["content"]
y_data = df["label"]

In [ ]:
# train/test split
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)

In [ ]:
# tf-idf vectorization
# ngram_range=(1,2) captures bigrams like "NOT_hate", "never bully"
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # unigrams + bigrams for negation context
    min_df=2,  # ignore very rare tokens
)

In [ ]:
tfidf_vectorizer.fit(x_train)

In [ ]:
x_train_encoded = tfidf_vectorizer.transform(x_train)
x_test_encoded = tfidf_vectorizer.transform(x_test)

In [ ]:
x_train_encoded.shape

In [ ]:
# model training and evaluation
import time
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC(),
    "MultinomialNB": MultinomialNB(),
    "SGD": SGDClassifier(),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(x_train_encoded, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(x_test_encoded)
    pred_time = time.time() - start

    results.append(
        [
            name,
            f1_score(y_test, y_pred, average="weighted"),
            precision_score(y_test, y_pred, average="weighted"),
            recall_score(y_test, y_pred, average="weighted"),
            train_time,
            pred_time,
        ]
    )

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "F1 Score",
        "Precision",
        "Recall",
        "Train Time",
        "Prediction Time",
    ],
)
print(results_df.sort_values("F1 Score", ascending=False).to_string(index=False))

Winner: SGDClassifier Reasons:Highest F1 score, very fast training, extremely fast prediction, works well with TF-IDF sparse features

In [ ]:
best_model = SGDClassifier()
best_model.fit(x_train_encoded, y_train)
y_pred_best = best_model.predict(x_test_encoded)

In [ ]:
print("\n── Classification Report (SGD) ──")
print(classification_report(y_test, y_pred_best))

In [ ]:
# save model and vextorizer
import joblib
 
joblib.dump(best_model, "model.pkl")
joblib.dump(tfidf_vectorizer, "vectorizer.pkl")
print("Model and vectorizer saved.")

In [ ]:
# Load saved model and vectorizer
model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

In [ ]:
# test some sentences
text = "Nobody likes you"

processedText = preprocess_text(text)
X = vectorizer.transform([processedText])
prediction = model.predict(X)[0]

if prediction == 1:
    print("Bullying detected")
else:
    print("Non-bullying content")

Testing sentences: 
- Nobody likes you
- Don't be so mean to people
- You're so stupid and ugly
- I don't like you
- fuck off you piece of shit